# Graph RAG and Multi-Modal RAG

This file teaches you RAG architectures that go beyond basic vector search.
By the end, you will know how Graph RAG, Multi-Modal RAG, Multi-Index RAG, RAPTOR, and Long-Context RAG work.

## Setup

In [1]:
!pip install python_dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

In [3]:
import json
import numpy as np
from openai import OpenAI

client = OpenAI(api_key = api_key)

In [4]:
EMBED_MODEL = "text-embedding-3-small"
MODEL = "gpt-4o-mini"

**What happened:** We imported libraries and set up model names.

In [6]:
def get_embeddings(texts):
    response = client.embeddings.create(
        model=EMBED_MODEL, 
        input=texts
    )
    
    embeddings = []
    for item in response.data:
        embeddings.append(item.embedding)
    return embeddings

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

**What happened:** `get_embeddings` converts text to vectors. `cosine_similarity` measures how close two vectors are.

## Graph RAG

**What it does:** Graph RAG builds a knowledge graph from documents. A knowledge graph is a collection of entities (things) and relationships (connections between things). Instead of searching text chunks, you search entities and their connections.

**When to use it:** Questions that need cross-document reasoning, or overview questions about themes in a large document set.

**Steps:**
1. Extract entities and relationships from text using the LLM
2. Store entities and relationships in a graph structure
3. At query time, find relevant entities and use their connections to answer

### Step 1: Extract Entities and Relationships

In [10]:
def extract_knowledge_graph(text):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=500, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content": (
                "Extract entities and relationships.\n\n"
                f"Text: {text}\n\n"
                'Return JSON: {"entities": '
                '[{"name": "...", "type": "..."}], '
                '"relationships": [{"subject": "...", '
                '"predicate": "...", "object": "..."}]}'
                )
            }
        ]
    )

    content = response.choices[0].message.content

    print(content)
    print(json.loads(content))
    
    graph = json.loads(content)
    return graph

**What happened:** This function sends text to the LLM and asks it to extract entities (like HPA, Prometheus) and relationships (like "HPA uses Prometheus").

In [13]:
doc = (
    "Kubernetes HPA uses Prometheus metrics to scale pods. "
    "HPA checks the metrics API every 15 seconds. "
    "KEDA extends HPA with event-driven scaling from "
    "sources like Kafka and Redis."
)

graph = extract_knowledge_graph(doc)

print("Entities:")
for i in range(len(graph['entities'])):
    entity = graph['entities'][i]
    name = entity['name']
    entity_type = entity['type']
    print(f"  [{entity_type}] {name}")

{
  "entities": [
    {
      "name": "Kubernetes HPA",
      "type": "System"
    },
    {
      "name": "Prometheus",
      "type": "Tool"
    },
    {
      "name": "metrics API",
      "type": "API"
    },
    {
      "name": "KEDA",
      "type": "Extension"
    },
    {
      "name": "Kafka",
      "type": "Event Source"
    },
    {
      "name": "Redis",
      "type": "Event Source"
    },
    {
      "name": "pods",
      "type": "Resource"
    }
  ],
  "relationships": [
    {
      "subject": "Kubernetes HPA",
      "predicate": "uses",
      "object": "Prometheus metrics"
    },
    {
      "subject": "Kubernetes HPA",
      "predicate": "checks",
      "object": "metrics API"
    },
    {
      "subject": "Kubernetes HPA",
      "predicate": "scales",
      "object": "pods"
    },
    {
      "subject": "KEDA",
      "predicate": "extends",
      "object": "Kubernetes HPA"
    },
    {
      "subject": "KEDA",
      "predicate": "provides",
      "object": "event-driven sc

**What happened:** The LLM extracted entities from the text. Each entity has a name and a type (like technology or concept).

In [12]:
print("Relationships:")
for i in range(len(graph['relationships'])):
    rel = graph['relationships'][i]
    subject = rel['subject']
    predicate = rel['predicate']
    obj = rel['object']
    print(f"  {subject} --{predicate}--> {obj}")

Relationships:
  Kubernetes HPA --uses--> Prometheus metrics
  Kubernetes HPA --checks--> metrics API
  metrics API --frequency--> every 15 seconds
  KEDA --extends--> Kubernetes HPA
  KEDA --provides--> event-driven scaling
  event-driven scaling --from sources--> Kafka
  event-driven scaling --from sources--> Redis


**What happened:** The LLM found connections between entities. Each relationship has a subject, a predicate (the connection type), and an object.

### Step 2: Query the Graph

In [15]:
def query_graph(question, graph):
    
    entities_json = json.dumps(graph['entities'])
    rels_json = json.dumps(graph['relationships'])

    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=200, 
        temperature=0,
        messages=[
            {
                "role": "user", 
                "content": (
                "Using this knowledge graph, answer:\n\n"
                f"Entities: {entities_json}\n"
                f"Rels: {rels_json}"
                f"\n\nQuestion: {question}"
                )
            }
        ]
    )
    
    answer = response.choices[0].message.content
    return answer

**What happened:** This function sends the graph (entities and relationships) to the LLM and asks it to answer a question using the graph.

In [16]:
answer = query_graph(
    "What is the relationship between HPA and KEDA?",
    graph
)
print(f"Answer: {answer}")

Answer: The relationship between Kubernetes HPA (Horizontal Pod Autoscaler) and KEDA (Kubernetes Event-driven Autoscaling) is that KEDA extends Kubernetes HPA. Additionally, KEDA provides event-driven scaling capabilities, which enhance the functionality of HPA.


**What happened:** The LLM used the graph relationships to explain how HPA and KEDA are connected.

## Multi-Modal RAG

**What it does:** Multi-Modal RAG retrieves both text and images. It handles documents that contain diagrams, charts, or photos alongside text.

**When to use it:** Product catalogs, technical manuals with diagrams, medical records.

**Steps:**
1. Convert images to text descriptions using a vision model (like GPT-4o)
2. Embed the descriptions alongside regular text
3. At query time, search across both text and image descriptions

### Three Approaches to Multi-Modal RAG

1. **Separate indexes** - One index for text, one for images. Search both at query time.
2. **Vision LLM descriptions** - Convert images to text descriptions. Index everything as text.
3. **Multi-modal embeddings** - Use a model like CLIP that can embed both text and images in the same vector space.

We will demo Approach 2 because it works with standard text embeddings.